<a href="https://colab.research.google.com/github/EthanEShea/SmartVineMonitoring/blob/main/LeaveExtraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Turns the hard to look at 360 videos, into clear grouped frames of leaves.

In [11]:
import cv2
import os

projectDir = "dataWine"
videoDir = os.path.join(projectDir, "Videos")
frameDir = os.path.join(projectDir, "Frames")
modelDir = os.path.join(projectDir, "models")
videoName = ""  # insert .mp4 here
videoPath = os.path.join(videoDir, videoName)

cap = cv2.VideoCapture(videoPath)
# Output path for the frames (update accordingly)
os.makedirs(frameDir, exist_ok=True)

# Creating frames from the video.
frameCount = 0
savedCount = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break

    if frameCount % 100 == 0: # adjust the frameCount as needed
        framePath = os.path.join(frameDir, f"walk_frame_{savedCount:05d}.jpg")
        cv2.imwrite(framePath, frame)
        print("Reading frame:", frameCount)
        savedCount += 1
    frameCount += 1

cap.release()
print("Frames processed:", frameCount)
print("Frames saved:", savedCount)

Frames processed: 0
Frames saved: 0


Used the images that were created from the video and used CVAT to create labeled data. Drew bounding boxes around leaves that were closer in the picture(to hopefully get a clearer picture). The model we trained for this worked for our videos, but more labeled images may be beneficial.

In [ ]:
data_yaml = f"""
path: {projectDir}
train: images/train
val: images/val

names:
  0: leafs
"""

with open(os.path.join(projectDir, "data.yaml"), "w") as f:
    f.write(data_yaml)

print("data.yaml created")

data.yaml created


In [ ]:
!pip install ultralytics
!yolo detect train data=data.yaml model=yolov8n.pt epochs=50 imgsz=640

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.

Runs the model, creates bounding boxes on the groups of leaves and then saves those bounding boxes to get a closer image of the leaves.

In [ ]:
# source is the same as outputPath (where our frames are)
# project is where the cropped images will go to
modelPath = os.path.join(modelDir, "best.pt")
resultsDir = os.path.join(projectDir, "yolo_results")
os.makedirs(resultsDir, exist_ok=True)

!yolo detect predict \
  model="{modelPath}" \
  source="{frameDir}" \
  save=True \
  save_crop=True \
  project="{resultsDir}" \
  name="predict_1" \
  exist_ok=True

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (NVIDIA RTX PRO 6000 Blackwell Server Edition, 97250MiB)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs

image 1/90 /content/drive/MyDrive/dataWine/Frames/walk_frame_00100.jpg: 320x640 1 leafs, 14.0ms
image 2/90 /content/drive/MyDrive/dataWine/Frames/walk_frame_00200.jpg: 320x640 2 leafss, 2.6ms
image 3/90 /content/drive/MyDrive/dataWine/Frames/walk_frame_00400.jpg: 320x640 2 leafss, 2.9ms
image 4/90 /content/drive/MyDrive/dataWine/Frames/walk_frame_00500.jpg: 320x640 2 leafss, 2.5ms
image 5/90 /content/drive/MyDrive/dataWine/Frames/walk_frame_00700.jpg: 320x640 1 leafs, 2.6ms
image 6/90 /content/drive/MyDrive/dataWine/Frames/walk_frame_00800.jpg: 320x640 2 leafss, 2.5ms
image 7/90 /content/drive/MyDrive/dataWine/Frames/walk_frame_01000.jpg: 320x640 2 leafss, 2.5ms
image 8/90 /content/drive/MyDrive/dataWine/Frames/walk_frame_01100.jpg: 320x640 2 leafss, 2.5ms
image 9/90 /content/drive/MyDrive